# 08 · Working with Dates and Times

**Goal:** learn to parse, extract, filter, and resample date/time data — one of the most
common real-world data types.

### Creating datetime data with `pd.to_datetime`

Dates loaded from CSV/Excel often arrive as plain strings. `pd.to_datetime` converts them into
pandas' proper `datetime64` dtype, unlocking date-aware operations.

In [1]:
import pandas as pd

dates_as_strings = ["2024-01-15", "2024-02-20", "2024-03-10"]
dates = pd.to_datetime(dates_as_strings)
print(dates)
print(dates.dtype)

DatetimeIndex(['2024-01-15', '2024-02-20', '2024-03-10'], dtype='datetime64[us]', freq=None)
datetime64[us]


In [2]:
# Works on a DataFrame column too, and handles multiple common formats automatically
df = pd.DataFrame({
    "order_date": ["2024-01-15", "2024-02-20", "2024-03-10", "2024-04-05"],
    "amount": [250, 300, 150, 400]
})

print(df.dtypes)
df["order_date"] = pd.to_datetime(df["order_date"])
print(df.dtypes)     # now a proper datetime64 dtype

order_date      str
amount        int64
dtype: object
order_date    datetime64[us]
amount                 int64
dtype: object


### The `.dt` accessor — extracting date parts

Just like `.str` unlocks string operations, `.dt` unlocks date/time operations on a datetime
Series.

In [3]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["day"] = df["order_date"].dt.day
df["day_name"] = df["order_date"].dt.day_name()      # e.g. "Monday"
df["month_name"] = df["order_date"].dt.month_name()   # e.g. "January"
df["quarter"] = df["order_date"].dt.quarter
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5   # Mon=0 ... Sun=6

print(df)

  order_date  amount  year  month  day day_name month_name  quarter  \
0 2024-01-15     250  2024      1   15   Monday    January        1   
1 2024-02-20     300  2024      2   20  Tuesday   February        1   
2 2024-03-10     150  2024      3   10   Sunday      March        1   
3 2024-04-05     400  2024      4    5   Friday      April        2   

   is_weekend  
0       False  
1       False  
2        True  
3       False  


### `pd.date_range` — generating sequences of dates

Very useful for building time series scaffolding (e.g. "every day in 2024") to merge real
data against.

In [4]:
print(pd.date_range(start="2024-01-01", end="2024-01-10"))     # daily by default
print()
print(pd.date_range(start="2024-01-01", periods=5, freq="D"))    # 5 days, from a start date
print()
print(pd.date_range(start="2024-01-01", periods=6, freq="ME"))    # 6 month-ends
print()
print(pd.date_range(start="2024-01-01", periods=4, freq="W"))     # 4 weeks

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10'],
              dtype='datetime64[us]', freq='D')

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05'],
              dtype='datetime64[us]', freq='D')

DatetimeIndex(['2024-01-31', '2024-02-29', '2024-03-31', '2024-04-30',
               '2024-05-31', '2024-06-30'],
              dtype='datetime64[us]', freq='ME')

DatetimeIndex(['2024-01-07', '2024-01-14', '2024-01-21', '2024-01-28'], dtype='datetime64[us]', freq='W-SUN')


### Common frequency codes

| Code | Meaning |
|---|---|
| `D` | Calendar day |
| `W` | Weekly |
| `ME` | Month end |
| `MS` | Month start |
| `QE` | Quarter end |
| `YE` | Year end |
| `h` | Hourly |
| `min` | Minutely |

### Using dates as the DataFrame index — enables date-based slicing

In [5]:
rng = pd.date_range(start="2024-01-01", periods=10, freq="D")
ts = pd.DataFrame({"value": range(100, 110)}, index=rng)
print(ts)

# With a DatetimeIndex, you can slice using natural date strings!
print()
print(ts["2024-01-03":"2024-01-06"])     # inclusive range, just like .loc label slicing
print()
print(ts.loc["2024-01"])                   # everything in January 2024 (partial-string indexing via .loc)

            value
2024-01-01    100
2024-01-02    101
2024-01-03    102
2024-01-04    103
2024-01-05    104
2024-01-06    105
2024-01-07    106
2024-01-08    107
2024-01-09    108
2024-01-10    109

            value
2024-01-03    102
2024-01-04    103
2024-01-05    104
2024-01-06    105

            value
2024-01-01    100
2024-01-02    101
2024-01-03    102
2024-01-04    103
2024-01-05    104
2024-01-06    105
2024-01-07    106
2024-01-08    107
2024-01-09    108
2024-01-10    109


### Filtering by date condition (when dates are a regular column, not the index)

In [6]:
df_filtered = df[df["order_date"] >= "2024-02-01"]
print(df_filtered[["order_date", "amount"]])

df_filtered2 = df[(df["order_date"] >= "2024-02-01") & (df["order_date"] <= "2024-03-31")]
print()
print(df_filtered2[["order_date", "amount"]])

  order_date  amount
1 2024-02-20     300
2 2024-03-10     150
3 2024-04-05     400

  order_date  amount
1 2024-02-20     300
2 2024-03-10     150


### Date arithmetic with `Timedelta`

In [7]:
df["due_date"] = df["order_date"] + pd.Timedelta(days=30)
print(df[["order_date", "due_date"]])

# Difference between two datetime columns gives a Timedelta
df["days_since_order"] = pd.Timestamp("2024-05-01") - df["order_date"]
print(df[["order_date", "days_since_order"]])

  order_date   due_date
0 2024-01-15 2024-02-14
1 2024-02-20 2024-03-21
2 2024-03-10 2024-04-09
3 2024-04-05 2024-05-05
  order_date days_since_order
0 2024-01-15         107 days
1 2024-02-20          71 days
2 2024-03-10          52 days
3 2024-04-05          26 days


### Resampling — aggregating time series data to a different frequency

`.resample()` is like `groupby()`, but specifically for time-based grouping (e.g. "daily data
→ monthly totals"). Requires a `DatetimeIndex`.

In [8]:
rng = pd.date_range(start="2024-01-01", periods=15, freq="D")
daily_sales = pd.DataFrame({"sales": [10, 12, 8, 15, 20, 18, 14, 9, 11, 16, 22, 19, 13, 17, 21]},
                             index=rng)
print(daily_sales.head())

weekly_totals = daily_sales.resample("W").sum()      # total sales per week
print()
print(weekly_totals)

weekly_avg = daily_sales.resample("W").mean()          # average daily sales per week
print()
print(weekly_avg)

            sales
2024-01-01     10
2024-01-02     12
2024-01-03      8
2024-01-04     15
2024-01-05     20

            sales
2024-01-07     97
2024-01-14    107
2024-01-21     21

                sales
2024-01-07  13.857143
2024-01-14  15.285714
2024-01-21  21.000000


### Rolling windows — moving averages

`.rolling(window)` computes a value over a sliding window — the classic example being a
moving average to smooth out day-to-day noise.

In [9]:
daily_sales["3day_avg"] = daily_sales["sales"].rolling(window=3).mean()
print(daily_sales)
# Notice the first 2 rows are NaN -- there aren't yet 3 days of data to average

            sales   3day_avg
2024-01-01     10        NaN
2024-01-02     12        NaN
2024-01-03      8  10.000000
2024-01-04     15  11.666667
2024-01-05     20  14.333333
2024-01-06     18  17.666667
2024-01-07     14  17.333333
2024-01-08      9  13.666667
2024-01-09     11  11.333333
2024-01-10     16  12.000000
2024-01-11     22  16.333333
2024-01-12     19  19.000000
2024-01-13     13  18.000000
2024-01-14     17  16.333333
2024-01-15     21  17.000000


### 🧠 Quick check

1. What does `pd.to_datetime()` do, and why is it necessary before doing date math?
2. What's the difference between `.resample()` and `.groupby()`?
3. Why does a 3-day rolling average have `NaN` for the first two rows?

<details>
<summary>Answers</summary>

1. It converts date strings (or other date-like values) into pandas' `datetime64` dtype,
   which is required to unlock `.dt` accessors, date arithmetic, and date-based slicing —
   plain strings don't support any of that.
2. `.resample()` groups specifically by time intervals (requires a datetime index/column) and
   understands calendar frequencies (weeks, months, etc.); `.groupby()` groups by the values
   of any column, not necessarily time-based.
3. A rolling window needs enough prior data points to compute — with `window=3`, the first
   two rows don't yet have 3 values available, so pandas can't compute an average for them.
</details>

### ✍️ Practice

1. Convert a column of date strings to `datetime64` and extract the day of week for each.
2. Create a `date_range` covering all of Q1 2024 (Jan–Mar) at daily frequency.
3. Given a DataFrame with a `DatetimeIndex` of daily data, resample it to monthly totals.
4. Compute a 7-day rolling average of a daily sales column.

Continue to **`09_io_and_pivot_tables.ipynb`** next.